##### How to convert (flatten) `array to string` using `concat_ws()`?

- **concat_ws()** = concatenate with `separator`
- It `flattens` an `array into a single string` by `joining elements` using a `delimiter`.

**Syntax**

     concat_ws(delimiter, column)

- **delimiter:** string like `",", "|", " "`.
- **column:** array column
  - `array_column:` The name of the column containing the `array data`.
  - syntax expect the `elements within the array` to be of a **string type (e.g., array<string>)**.
  - If the elements are `not strings` (e.g., `array<int> or array<long>)`, you may need to explicitly `cast` them to `string type` first.

In [0]:
data = [
    (1, ["Java", "Python", "Spark", "sparkSQL"]),
    (2, ["Java", "", "Spark", None]),
    (3, ["SQL", "Excel", ""]),
    (4, ["Scala", "", "R"]),
    (5, ["SQL", "PySpark", "Java"]),
    (6, ["Airflow", "Devops", "SparkSQL"]),
    (7, ["Azure", None, "Engineer"]),
    (8, ["", None, "Engineer"]),
    (9, ["", "", ""]),
    (10, [None, None, None]),
    (11, [None, None, "SQL"]),
    (12, [None, "AWS", None]),
    (13, ["", "", "Airflow"]),
    (14, [])
]

df_arry = spark.createDataFrame(data, ["id", "skills"])
display(df_arry)

id,skills
1,"List(Java, Python, Spark, sparkSQL)"
2,"List(Java, , Spark, null)"
3,"List(SQL, Excel, )"
4,"List(Scala, , R)"
5,"List(SQL, PySpark, Java)"
6,"List(Airflow, Devops, SparkSQL)"
7,"List(Azure, null, Engineer)"
8,"List(, null, Engineer)"
9,"List(, , )"
10,"List(null, null, null)"


In [0]:
from pyspark.sql.functions import concat_ws, col, lit

- `Ignores NULL` values inside `array`.
- If `array` itself is `NULL`: `result is NULL`.
- Keeps `empty strings ("")` as-is.

In [0]:
df_arry_conc = (df_arry \
    .withColumn("flatten_comma", concat_ws(", ", "skills"))
    .withColumn("flatten_empty", concat_ws("", "skills"))        # Separator = "" (empty string); Elements are directly glued together
    .withColumn("flatten_backlash", concat_ws(" / ", "skills"))
    .withColumn("flatten_semicolon", concat_ws("; ", "skills"))
    .withColumn("flatten_pipeline", concat_ws(" | ", "skills")))
display(df_arry_conc)

id,skills,flatten_comma,flatten_empty,flatten_backlash,flatten_semicolon,flatten_pipeline
1,"List(Java, Python, Spark, sparkSQL)","Java, Python, Spark, sparkSQL",JavaPythonSparksparkSQL,Java / Python / Spark / sparkSQL,Java; Python; Spark; sparkSQL,Java | Python | Spark | sparkSQL
2,"List(Java, , Spark, null)","Java, , Spark",JavaSpark,Java / / Spark,Java; ; Spark,Java | | Spark
3,"List(SQL, Excel, )","SQL, Excel,",SQLExcel,SQL / Excel /,SQL; Excel;,SQL | Excel |
4,"List(Scala, , R)","Scala, , R",ScalaR,Scala / / R,Scala; ; R,Scala | | R
5,"List(SQL, PySpark, Java)","SQL, PySpark, Java",SQLPySparkJava,SQL / PySpark / Java,SQL; PySpark; Java,SQL | PySpark | Java
6,"List(Airflow, Devops, SparkSQL)","Airflow, Devops, SparkSQL",AirflowDevopsSparkSQL,Airflow / Devops / SparkSQL,Airflow; Devops; SparkSQL,Airflow | Devops | SparkSQL
7,"List(Azure, null, Engineer)","Azure, Engineer",AzureEngineer,Azure / Engineer,Azure; Engineer,Azure | Engineer
8,"List(, null, Engineer)",", Engineer",Engineer,/ Engineer,; Engineer,| Engineer
9,"List(, , )",", ,",,/ /,; ;,| |
10,"List(null, null, null)",,,,,


     # Array elements
     ["Scala", "", "R"]

     # Separator
     ""   (nothing)

     # Joining happens like this
     "Scala" + "" + "" + "" + "R"

     # Final Result
     ScalaR

- `"Scala"`
- `separator "" → nothing`
- `"" → contributes nothing`
- `"R"`

|    Type                        |           Input               |     Output       |
|--------------------------------|-------------------------------|------------------|
| NULL is ignored                | ["Azure", None, "Engineer"]   |  Azure, Engineer |
| Empty string "" is NOT ignored | ["Scala", "", "R"]            |  Scala, , R      |
| Empty array [] → empty string  | []                            |  ""              |

| id | flatten_comma                 | flatten_empty           | flatten_backlash                 | flatten_semicolon             | flatten_pipeline                 |
| -- | ----------------------------- | ----------------------- | -------------------------------- | ----------------------------- | -------------------------------- |
| 14  | (empty string)                | (empty string)          | (empty string)                   | (empty string)                | (empty string)                   |